In [2]:
!pip install tensorflow

  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
  Using cached werkzeug-3.1.4-py3-none-any.whl.metadata (4.0 kB)
  Using cached wheel-0.45.1-py3-none-any.whl.metadata (2.3 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/331.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/331.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/331.7 MB ? eta -:--:--
   --------------------------

In [5]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from scipy.stats import multivariate_normal
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [6]:
base = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')
feature_extractor = Model(inputs=base.input, outputs=base.output)

C:\Users\HP\AppData\Local\Temp\ipykernel_12844\1230895191.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 25s 3us/step


In [ ]:
datagen = ImageDataGenerator(rescale=1./255)

train_gen = datagen.flow_from_directory(
    'SkinDisease/train',
    target_size=(320,320),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

test_gen = datagen.flow_from_directory(
    'SkinDisease/test',
    target_size=(320,320),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

X_train = feature_extractor.predict(train_gen)
y_train = train_gen.classes
num_classes = len(np.unique(y_train))

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'dataset/train'

In [ ]:
class GaussianBayes:
    def __init__(self):
        self.mean = {}
        self.cov = {}
        self.prior = {}

    def fit(self, X, y):
        for c in np.unique(y):
            Xc = X[y == c]
            self.mean[c] = np.mean(Xc, axis=0)
            self.cov[c] = np.cov(Xc.T) + np.eye(X.shape[1]) * 1e-6
            self.prior[c] = len(Xc) / len(X)

In [ ]:
def predict(self, X):
    preds = []
    probs = []

    for x in X:
        posteriors = {}
        for c in self.mean:
            likelihood = multivariate_normal.pdf(x, self.mean[c], self.cov[c])
            posterior = likelihood * self.prior[c]
            posteriors[c] = posterior
        
        total = sum(posteriors.values())
        posteriors = {k:v/total for k,v in posteriors.items()}

        best = max(posteriors, key=posteriors.get)
        preds.append(best)
        probs.append(posteriors)

    return np.array(preds), probs

In [ ]:
model = GaussianBayes()
model.fit(X_train, y_train)

X_test = feature_extractor.predict(test_gen)
y_test = test_gen.classes

pred, prob = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

print("Accuracy:", accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))

In [ ]:
def top3(probs):
    return sorted(probs.items(), key=lambda x: x[1], reverse=True)[:3]

for i in range(5):
    print("Top 3 diagnosis:", top3(prob[i]))